In [ ]:
# ============================================================
#  REGRESSION TECHNIQUES — COMPLETE IMPLEMENTATION
#  Problem: Implement Regression Technique & Evaluate Performance
# ============================================================

# ─────────────────────────────────────────────
# STEP 1: Importing Python Modules / Libraries
# ─────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    LinearRegression, LogisticRegression, Lasso, Ridge
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully.\n")


# ─────────────────────────────────────────────
# STEP 2: Importing & Displaying Data
# ─────────────────────────────────────────────
print("=" * 55)
print("STEP 2 — Loading & Displaying Data")
print("=" * 55)

# Load California Housing dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Introduce some missing values for demonstration
np.random.seed(42)
missing_idx = np.random.choice(df.index, size=50, replace=False)
df.loc[missing_idx[:25], "HouseAge"]     = np.nan
df.loc[missing_idx[25:], "AveBedrms"]   = np.nan

print(f"\nDataset shape : {df.shape}")
print(f"Columns       : {list(df.columns)}")
print("\n--- First 5 rows ---")
print(df.head())
print("\n--- Data types ---")
print(df.dtypes)


# ─────────────────────────────────────────────
# STEP 3: Statistical Analysis & Outlier Analysis
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 3 — Statistical Analysis & Outlier Analysis")
print("=" * 55)

print("\n--- Descriptive Statistics ---")
print(df.describe().round(2))

# Missing value summary
print("\n--- Missing Values ---")
missing = df.isnull().sum()
print(missing[missing > 0])

# Z-score based outlier detection
print("\n--- Outlier Detection (|Z-score| > 3) ---")
numeric_cols = df.select_dtypes(include=[np.number]).columns
z_scores = np.abs(stats.zscore(df[numeric_cols].dropna()))
outlier_counts = (z_scores > 3).sum(axis=0)
for col, cnt in zip(numeric_cols, outlier_counts):
    if cnt > 0:
        print(f"  {col}: {cnt} outliers")

# Correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap — California Housing")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=120)
plt.show()
print("  [Saved] correlation_heatmap.png")


# ─────────────────────────────────────────────
# STEP 4: Creating Independent & Dependent Variables
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 4 — Creating Independent & Dependent Variables")
print("=" * 55)

feature_cols = [c for c in df.columns if c != "MedHouseVal"]
X = df[feature_cols]
y = df["MedHouseVal"]   # Target (median house value)

print(f"\nFeatures  (X) : {feature_cols}")
print(f"Target    (y) : MedHouseVal  |  shape: {y.shape}")


# ─────────────────────────────────────────────
# STEP 5: Replacing Missing Values with Meaningful Values
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 5 — Handling Missing Values")
print("=" * 55)

print(f"\nMissing before imputation:\n{X.isnull().sum()[X.isnull().sum() > 0]}")

# Fill numeric columns with column median
X = X.fillna(X.median(numeric_only=True))

print(f"\nMissing after imputation:\n{X.isnull().sum()[X.isnull().sum() > 0].to_string() or '  None — all values filled ✅'}")


# ─────────────────────────────────────────────
# STEP 6: Splitting Data into Training & Test Sets
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 6 — Train / Test Split")
print("=" * 55)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\n  Train size : {X_train.shape[0]} samples")
print(f"  Test  size : {X_test.shape[0]} samples")

# Feature scaling
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("\n  Features scaled with StandardScaler ✅")


# ─────────────────────────────────────────────
# STEP 7: Apply Regression Techniques & Evaluate Performance
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 7 — Regression Models & Performance Evaluation")
print("=" * 55)

# Helper function for regression metrics
def regression_metrics(name, y_true, y_pred):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"\n  ── {name} ──")
    print(f"    MSE  : {mse:.4f}")
    print(f"    RMSE : {rmse:.4f}")
    print(f"    MAE  : {mae:.4f}")
    print(f"    R²   : {r2:.4f}")
    return {"Model": name, "MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}


results = []

# ── 7a. Linear Regression ────────────────────
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
results.append(regression_metrics("Linear Regression", y_test, y_pred_lr))

print("\n  Top feature coefficients:")
coef_df = pd.DataFrame({"Feature": X.columns, "Coefficient": lr.coef_})
print(coef_df.reindex(coef_df["Coefficient"].abs().sort_values(ascending=False).index)
      .head(5).to_string(index=False))


# ── 7b. Ridge Regression (L2) ────────────────
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)
results.append(regression_metrics("Ridge Regression (L2)", y_test, y_pred_ridge))


# ── 7c. LASSO Regression (L1) ────────────────
lasso = Lasso(alpha=0.01, max_iter=5000)
lasso.fit(X_train_sc, y_train)
y_pred_lasso = lasso.predict(X_test_sc)
results.append(regression_metrics("LASSO Regression (L1)", y_test, y_pred_lasso))

zero_coef = np.sum(lasso.coef_ == 0)
print(f"\n  LASSO zeroed-out {zero_coef} / {len(lasso.coef_)} features (automatic feature selection)")


# ── 7d. Logistic Regression (classification) ─
print("\n  [Logistic Regression — binary classification task]")
# Convert to binary: High value (>= median) = 1, else 0
threshold    = y.median()
y_bin        = (y >= threshold).astype(int)
y_bin_train  = (y_train >= threshold).astype(int)
y_bin_test   = (y_test  >= threshold).astype(int)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_sc, y_bin_train)
y_pred_log = log_reg.predict(X_test_sc)

print(f"\n  ── Logistic Regression ──")
print(f"    Accuracy : {accuracy_score(y_bin_test, y_pred_log):.4f}")
print("\n  Classification Report:")
print(classification_report(y_bin_test, y_pred_log,
      target_names=["Low Value", "High Value"]))

# Confusion matrix plot
cm = confusion_matrix(y_bin_test, y_pred_log)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low", "High"], yticklabels=["Low", "High"])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.show()
print("  [Saved] confusion_matrix.png")


# ─────────────────────────────────────────────
# Comparison Summary Table
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("MODEL COMPARISON SUMMARY (Linear/Ridge/LASSO)")
print("=" * 55)
summary = pd.DataFrame(results).set_index("Model")
print(summary.round(4).to_string())


# ─────────────────────────────────────────────
# Actual vs Predicted Plot (best model = Ridge)
# ─────────────────────────────────────────────
best_model_name = summary["R2"].idxmax()
print(f"\nBest model by R² : {best_model_name}")

preds_map = {
    "Linear Regression"   : y_pred_lr,
    "Ridge Regression (L2)": y_pred_ridge,
    "LASSO Regression (L1)": y_pred_lasso,
}
y_pred_best = preds_map[best_model_name]

plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred_best, alpha=0.3, s=15, color="steelblue")
lim = [y_test.min(), y_test.max()]
plt.plot(lim, lim, "r--", lw=1.5, label="Perfect prediction")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.legend()
plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=120)
plt.show()
print("[Saved] actual_vs_predicted.png")

print("\n✅ All steps completed successfully!")